# NOTEBOOK PEMISAHAN DATASET (3 RAS KUCING)
## Ekstraksi dan Pemisahan Khusus: Birman, Ragdoll, dan Siamese
---
Notebook ini memproses dataset **Oxford-IIIT Pet Dataset** (total 7.393 gambar)
untuk memfilter, memisahkan, dan menyusun **dataset baru tersendiri** khusus 3 ras target (*Birman, Ragdoll, Siamese*).

### Struktur Output yang Dihasilkan:
```text
dataset_3_ras/
├── all/                   (600 gambar: 200 Birman, 200 Ragdoll, 200 Siamese)
│   ├── Birman/
│   ├── Ragdoll/
│   └── Siamese/
├── train/                 (480 gambar: 160 per kelas - Stratified 80%)
│   ├── Birman/
│   ├── Ragdoll/
│   └── Siamese/
├── test/                  (120 gambar: 40 per kelas - Stratified 20%)
│   ├── Birman/
│   ├── Ragdoll/
│   └── Siamese/
└── metadata_dataset.csv   (Metadata lengkap file, ras, dan split)
```

---
### TAHAP 1 — IMPORT LIBRARY & PENGATURAN PARAMETER
Menyiapkan seluruh pustaka yang diperlukan dan menentukan parameter ras target.

In [ ]:
import os
import re
import glob
import shutil
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

# Konfigurasi Direktori dan Target
DATASET_SRC = "images"              # Folder sumber (Oxford-IIIT Pet)
TARGET_BREEDS = ["Birman", "Ragdoll", "Siamese"]
OUTPUT_DIR = "dataset_3_ras"        # Folder dataset tujuan
RANDOM_STATE = 42                   # Seed pengacakan agar hasil konsisten
TEST_SIZE = 0.20                    # 80% Data Latih, 20% Data Uji

print(f"Direktori sumber : '{DATASET_SRC}'")
print(f"Target ras       : {TARGET_BREEDS}")
print(f"Direktori output : '{OUTPUT_DIR}'")

---
### TAHAP 2 — SCANNING & FILTER GAMBAR 3 RAS TARGET
Membaca seluruh file pada folder `images/`, mengekstrak label menggunakan regex, memverifikasi integritas file gambar dengan PIL, dan menyusunnya ke dalam Pandas DataFrame.

In [ ]:
# Ambil seluruh path file gambar
all_files = glob.glob(os.path.join(DATASET_SRC, "*.*"))
print(f"Total file ditemukan pada folder sumber: {len(all_files)}")

# Pola regex Oxford-IIIT Pet: NamaRas_Angka.ekstensi
regex_pattern = re.compile(r"^([a-zA-Z_]+)_\d+\.(?:jpg|jpeg|png)$", re.IGNORECASE)

records = []
corrupted_files = []

for path in all_files:
    filename = os.path.basename(path)
    match = regex_pattern.match(filename)
    if match:
        breed = match.group(1)
        if breed in TARGET_BREEDS:
            try:
                # Uji apakah file gambar valid dan tidak rusak
                with Image.open(path) as img:
                    img.verify()
                records.append({
                    "original_path": path,
                    "filename": filename,
                    "breed": breed
                })
            except Exception as e:
                corrupted_files.append((filename, str(e)))

df_dataset = pd.DataFrame(records)
print(f"Jumlah gambar valid untuk 3 ras target : {len(df_dataset)}")
print(f"Jumlah gambar rusak dilewati           : {len(corrupted_files)}\n")

print("--- DISTRIBUSI JUMLAH PER KELAS ---")
for breed in TARGET_BREEDS:
    count = (df_dataset["breed"] == breed).sum()
    print(f" * {breed:<10}: {count} gambar")

df_dataset.head()

---
### TAHAP 3 — PEMBAGIAN DATA TERSTRATIFIKASI (80% TRAIN, 20% TEST)
Membagi dataset secara terstratifikasi (*stratified*) sehingga proporsi kelas antara data latih dan data uji tetap seimbang (160:40 per ras).

In [ ]:
train_df, test_df = train_test_split(
    df_dataset,
    test_size=TEST_SIZE,
    stratify=df_dataset["breed"],
    random_state=RANDOM_STATE
)

train_df = train_df.copy()
test_df = test_df.copy()
train_df["split"] = "train"
test_df["split"] = "test"

df_final = pd.concat([train_df, test_df], ignore_index=True)

# Tampilkan tabel rekapitulasi split
summary_split = pd.crosstab(df_final["breed"], df_final["split"], margins=True)
print("=" * 55)
print("REKAPITULASI PEMBAGIAN DATASET (STRATIFIED 80:20)")
print("=" * 55)
print(summary_split)
print("=" * 55)

---
### TAHAP 4 — MEMBUAT STRUKTUR DIREKTORI OUTPUT
Membuat hierarki folder untuk kategori `all/`, `train/`, dan `test/` per subfolder ras.

In [ ]:
folders = []
for breed in TARGET_BREEDS:
    folders.append(os.path.join(OUTPUT_DIR, "all", breed))
    folders.append(os.path.join(OUTPUT_DIR, "train", breed))
    folders.append(os.path.join(OUTPUT_DIR, "test", breed))

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print(f"Seluruh direktori berhasil dibuat di: '{os.path.abspath(OUTPUT_DIR)}'")

---
### TAHAP 5 — MENYALIN FILE GAMBAR & MENYIMPAN METADATA CSV
Menyalin file gambar asli ke folder tujuan dan membuat rekaman metadata CSV yang memetakan setiap file ke ras dan splitnya.

In [ ]:
metadata_records = []

print("Sedang menyalin file gambar ke folder dataset baru...")
for idx, row in df_final.iterrows():
    src = row["original_path"]
    fn = row["filename"]
    breed = row["breed"]
    split = row["split"]

    # 1. Simpan ke all/{breed}/
    dest_all = os.path.join(OUTPUT_DIR, "all", breed, fn)
    shutil.copy2(src, dest_all)

    # 2. Simpan ke {split}/{breed}/
    dest_split = os.path.join(OUTPUT_DIR, split, breed, fn)
    shutil.copy2(src, dest_split)

    metadata_records.append({
        "filename": fn,
        "breed": breed,
        "split": split,
        "path_all": dest_all,
        "path_split": dest_split
    })

# Simpan metadata CSV
df_meta = pd.DataFrame(metadata_records)
csv_path = os.path.join(OUTPUT_DIR, "metadata_dataset.csv")
df_meta.to_csv(csv_path, index=False)

print(f"Penyalinan selesai!")
print(f"File metadata CSV tersimpan di: '{csv_path}'")
df_meta.head()

---
### TAHAP 6 — VERIFIKASI AKHIR JUMLAH FILE
Memastikan bahwa setiap folder memiliki jumlah file yang tepat dan sesuai target.

In [ ]:
print("=" * 60)
print("VERIFIKASI JUMLAH FILE PADA DATASET BARU")
print("=" * 60)

for category in ["all", "train", "test"]:
    total_cat = 0
    print(f"\nKategori: {category.upper()}")
    for breed in TARGET_BREEDS:
        folder_path = os.path.join(OUTPUT_DIR, category, breed)
        count = len(os.listdir(folder_path))
        total_cat += count
        print(f"  * {breed:<10}: {count} gambar")
    print(f"  Total {category}: {total_cat} gambar")

print("\n" + "=" * 60)
print("SUKSES! DATASET KHUSUS 3 RAS KUCING SIAP DIGUNAKAN!")
print("=" * 60)